# Démonstration Colab — Détection de fraude à l'assurance habitation

**Variante FULL : CLIP + BLIP-2 + LLaVA + Mistral-7B-Instruct**

Projet Data Mining — ISFA, Promotion 2025-2026

Équipe : DOSSOU Gypsie · Maty GUEYE · Arthur Pédro HOUNGBEDJI

---

## À quoi sert ce notebook

Ce notebook déploie l'application Streamlit en mode **full** (avec les 3 LMM/LLM open-source). L'app est exposée via un tunnel **ngrok** pour produire une URL publique accessible depuis n'importe quel navigateur.

**Prérequis :**

- Un compte HuggingFace gratuit et un **token Read** : https://huggingface.co/settings/tokens
- Un compte ngrok gratuit et un **authtoken** : https://dashboard.ngrok.com/get-started/your-authtoken
- Un runtime Colab GPU : **L4** (24 Go VRAM) ou **A100** recommandés.

**Configurer le runtime :** Menu *Exécution > Modifier le type d'exécution > GPU > L4 ou A100 > Enregistrer.*

## 1. Vérification du GPUVérifions que le runtime est bien configuré en GPU et qu'il y a au moins 24 Go de VRAM disponibles.

In [ ]:
!nvidia-smi

## 2. Clonage du dépôt GitHub

Le code source du projet est public sur GitHub. On le clone dans `/content`.

In [ ]:
import os
os.chdir('/content')

REPO_URL = "https://github.com/HpedArthur/Insurance-fraud-detection-AI.git"
REPO_DIR = "/content/Insurance-fraud-detection-AI"

!rm -rf {REPO_DIR}
!git clone {REPO_URL}

os.chdir(REPO_DIR)
print('\n=== Contenu du dossier 04_models/ ===')
!ls -lh 04_models/

## 3. Installation des dépendances

Colab a déjà PyTorch et transformers. On ajoute uniquement ce qui manque pour notre pipeline :

- `streamlit` (interface)
- `open_clip_torch` (encodeur CLIP)
- `bitsandbytes` (quantization 4-bit, optionnel — fallback fp16 si cassé)
- `imagehash`, `imbalanced-learn`, `xgboost`, `shap` (pipeline ML)
- `spacy` + le modèle français `fr_core_news_md`
- `pyngrok` (tunnel public)

Compter environ **3 à 5 minutes** pour cette cellule.

In [ ]:
!pip install -q streamlit==1.42.0 open_clip_torch \
    transformers accelerate sentencepiece protobuf \
    imagehash imbalanced-learn xgboost shap \
    spacy==3.8.3 pyngrok

!pip install -q https://github.com/explosion/spacy-models/releases/download/fr_core_news_md-3.8.0/fr_core_news_md-3.8.0-py3-none-any.whl

# bitsandbytes (peut echouer sur Colab Python 3.12 + CUDA 12.8 — le code a un fallback fp16)
!pip install -q --upgrade bitsandbytes

print('\nInstallation terminee.')

## 4. Authentification HuggingFace

Mistral-7B et LLaVA-1.5-7B nécessitent un compte HuggingFace pour le téléchargement.

**Avant la première exécution, accepter les licences sur ces deux pages :**

- https://huggingface.co/llava-hf/llava-1.5-7b-hf (cliquer « Agree and access repository » si nécessaire)
- https://huggingface.co/mistralai/Mistral-7B-Instruct-v0.3 (idem)

Puis générer un token **Read** sur https://huggingface.co/settings/tokens et le coller ci-dessous.

**Important** : ne jamais partager ce notebook avec votre token écrit dedans. Effacez-le avant de transmettre le fichier.

In [ ]:
from huggingface_hub import login

# Remplacer la valeur entre guillemets par votre token Read (format hf_XXXX...)
HF_TOKEN = "hf_REMPLACEZ_PAR_VOTRE_TOKEN"

if HF_TOKEN.startswith("hf_") and HF_TOKEN != "hf_REMPLACEZ_PAR_VOTRE_TOKEN":
    login(token=HF_TOKEN)
    print('Authentification HuggingFace reussie.')
else:
    print('Token HuggingFace non configure.')
    print('Remplacez la valeur de HF_TOKEN par votre token Read puis relancez la cellule.')

## 5. Configuration du tunnel ngrok

ngrok expose le port local 8501 de Streamlit sur une URL publique accessible depuis n'importe quel navigateur.

**Récupérer son authtoken :** https://dashboard.ngrok.com/get-started/your-authtoken (compte gratuit suffit).

In [ ]:
from pyngrok import ngrok, conf

# Remplacer la valeur entre guillemets par votre authtoken ngrok
NGROK_TOKEN = "REMPLACEZ_PAR_VOTRE_AUTHTOKEN_NGROK"

if NGROK_TOKEN != "REMPLACEZ_PAR_VOTRE_AUTHTOKEN_NGROK":
    conf.get_default().auth_token = NGROK_TOKEN
    print('Authtoken ngrok configure.')
else:
    print('Authtoken ngrok non configure.')
    print('Remplacez la valeur de NGROK_TOKEN par votre authtoken puis relancez la cellule.')

## 6. Lancement de Streamlit et du tunnel ngrok

Cette cellule fait trois choses :

1. Tue toute instance Streamlit en cours et tout tunnel ngrok actif.
2. Lance Streamlit en arrière-plan sur le port 8501, avec les options compatibles tunnel (CORS et XSRF désactivés).
3. Ouvre un tunnel ngrok et affiche l'URL publique.

**Cliquer sur l'URL `https://xxxx.ngrok-free.app` qui s'affiche** pour accéder à l'app. ngrok affichera une page d'accueil au premier passage : cliquer simplement sur « Visit Site ».

In [ ]:
import subprocess
import time
import os
from pyngrok import ngrok

# 1. Nettoyage
subprocess.run(['pkill', '-f', 'streamlit'], capture_output=True)
ngrok.kill()
time.sleep(2)

# 2. Lancement de Streamlit
os.chdir('/content/Insurance-fraud-detection-AI')
log_path = '/content/streamlit.log'

subprocess.Popen(
    ['streamlit', 'run', '05_app/app.py',
     '--server.port', '8501',
     '--server.headless', 'true',
     '--server.fileWatcherType', 'none',
     '--server.enableCORS', 'false',
     '--server.enableXsrfProtection', 'false',
     '--browser.gatherUsageStats', 'false'],
    stdout=open(log_path, 'w'), stderr=subprocess.STDOUT,
)

print('Streamlit demarre... (attente 15 secondes)')
time.sleep(15)

# 3. Tunnel ngrok
public_url = ngrok.connect(8501, 'http')
url = public_url.public_url if hasattr(public_url, 'public_url') else str(public_url)

print('\n' + '=' * 60)
print(f'  URL publique :  {url}')
print('=' * 60)
print('\nOuvrez cette URL dans un nouvel onglet.')
print('Au premier acces, ngrok affiche une page de bienvenue : cliquer « Visit Site ».')

## 7. Mode d'emploi de l'application

Une fois sur l'URL ngrok, l'application Streamlit est ouverte.

**Configuration recommandée pour la démonstration full :**

1. Dans la barre latérale (sidebar) :
   - **Variante de modèle** : sélectionner **full**
   - **Modèles avancés** : cocher **BLIP-2 + LLaVA (image)** et **Mistral judge (texte)**

2. Remplir le formulaire (le texte des circonstances est pré-rempli avec un exemple plausible).

3. **Téléverser une photo de sinistre** dans la section *Photo du sinistre*.

4. Cliquer **Soumettre la déclaration**.

**Premier upload : compter 3 à 6 minutes** car BLIP-2 et Mistral sont téléchargés depuis HuggingFace Hub (environ 20 Go au total). Les uploads suivants prennent 20 à 40 secondes.

L'application affiche alors le score image, le score multimodal, la heatmap d'occlusion, la décision à trois niveaux (Légitime / À expertiser / Fraude probable) et le détail de toutes les features extraites.

---

## Diagnostic et résolution des problèmes

**Pour consulter les logs Streamlit en cas de comportement inattendu :**

In [ ]:
!tail -n 60 /content/streamlit.log

**Problèmes connus et solutions :**

**bitsandbytes cassé** (cas typique sur Colab Python 3.12 + CUDA 12.8 — message `No module named 'triton.ops'`).
Le code détecte automatiquement cette situation et bascule en fallback : Mistral est chargé en fp16 (14 Go au lieu de 5 Go en 4-bit), et LLaVA est désactivé pour rester dans le budget VRAM. BLIP-2 reste actif et fournit déjà 3 scores forensiques. Le log doit afficher la ligne `Mistral charge en fp16 (sans bitsandbytes)`.

**CUDA out of memory.**
Vérifier que le runtime est sur **L4 (24 Go)** ou **A100**. Sur T4 (16 Go), décocher *BLIP-2 + LLaVA* dans la sidebar et ne garder que *Mistral judge*.

**Erreur 401 lors du téléchargement de Mistral.**
Vérifier que la cellule 4 (authentification HuggingFace) a bien été exécutée avec un token valide, et que la licence du modèle a été acceptée sur la page du modèle.

**URL ngrok inaccessible (502 Bad Gateway).**
Streamlit n'a pas eu le temps de démarrer. Relancer la cellule 6.

**Pour relancer proprement Streamlit sans tout réinstaller**, exécuter simplement la cellule 6 à nouveau : elle commence par tuer l'instance précédente.

---

## Alternative — application sans GPU

La variante **lite** (CLIP + EXIF + features handcrafted texte) est déployée publiquement sur HuggingFace Spaces, sans dépendance GPU. Elle est accessible directement à :

https://huggingface.co/spaces/ArthurPedro/insurance-fraud-detection

Les performances comparées des deux variantes sont détaillées dans le rapport (sections 4.1 et 4.2).